                # BetterTogether (Apple Silicon / MPS)

                Alternate prompt optimization and weight optimization, evaluate intermediate programs, and retain the best strategy stage.

                **Use it when:** You have both a useful prompt optimizer and a trainable local model, and want them to improve each other rather than run in isolation.

                **What compilation changes:** Prompt demonstrations first, then a Qwen LoRA adapter in the explicit `p -> w` candidate.

                Important configuration in this benchmark:

                - BootstrapFewShotWithRandomSearch (`p`) plus BootstrapFinetune (`w`)
- stock DSPy BetterTogether with explicit `p -> w` strategy
- MPS-backed Qwen2.5-0.5B-Instruct, 18 weight steps, seed 42
- preserve original, `p`, and `p -> w` candidates even when validation ties

                The 74-row dataset and pair-grouped train/validation/test membership are frozen.
                The test partition is deliberately baseline-adversarial, so these scores teach
                optimizer tradeoffs; they are not a general-purpose AI-detector leaderboard.

In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "benchmark_summary.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import (
    artifact_paths,
    benchmark_snapshot,
    learned_program_preview,
    verify_prompt_artifact,
)

OPTIMIZER = 'better-together'
print(f"optimizer={OPTIMIZER!r}")
print("reading the checked-in chapter result; no API calls")

optimizer='better-together'
reading the checked-in chapter result; no API calls


                ## Compile shape

                This is the essential DSPy call used by the shared runner (setup variables omitted):

                ```python
                optimizer = dspy.BetterTogether(
    metric=exact_match, p=prompt_optimizer, w=weight_optimizer,
)
optimized_detector = optimizer.compile(
    detector, trainset=trainset, teacher=local_teacher, valset=valset,
    strategy='p -> w', seed=42,
)
                ```

                `compile` returns a program. Calling that program on the untouched test examples is
                a separate phase; the notebook reports optimization cost/time separately from inference latency.

In [2]:
print(benchmark_snapshot(OPTIMIZER))
print()
print(artifact_paths(OPTIMIZER))

BetterTogether — frozen full-profile rerun
status: completed
task model: Qwen/Qwen2.5-0.5B-Instruct
test accuracy: 50.0%
uplift: +0.0 points vs its Qwen run baseline
optimization: $0.0000 and 4.6min
inference latency: mean 1.39s; p95 1.94s
reload checks: prompt=True; model=True; predictions=3/3 labels
comparison boundary: same frozen split, separate Qwen/MPS experiment; not Luna-comparable

Published artifacts:
- canonical program snapshot: chapter06/optimized_programs/final/better-together.json
- canonical prompt: chapter06/results/final_prompts/better-together.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

DSPy retained the original program when all validation candidates tied. That is not a failed run: inspect `candidate_programs/` to see the completed prompt-only and trained `p -> w` alternatives and the adapter that was deliberately preserved.

The next cell shows a bounded readable preview. The complete, lossless prompt and
saved program snapshot remain at the paths printed above.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (59 characters):
Decide whether the supplied passage was generated by an AI.

Demonstrations: 0

Frozen program/prompt consistency: {'checked': True, 'predictors': 1, 'prompt_state_equal': True, 'mismatches': []}


## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Evaluate the returned program on a test set that was not
used during compilation, and compare accuracy, compile cost, and inference latency
rather than treating a single score as the whole result.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`. Raw provider
transcripts and temporary training outputs are intentionally excluded from the
student download.